In [1]:
import cv2
import os
import numpy as np
from ultralytics import YOLO

# -------------------------------------------------------------------
# !!! CẤU HÌNH BẮT BUỘC !!!
# (Bạn PHẢI thay đổi các giá trị này cho phù hợp với phiếu của bạn)
# -------------------------------------------------------------------

# --- 1. CẤU HÌNH FILE VÀ MODEL ---
MODEL_PATH = 'best.pt'
INPUT_IMAGE_PATH = r"D:\data\phieu_50.jpg" # Tên ảnh phiếu trắc nghiệm
CONFIDENCE_THRESHOLD = 0.4 # Ngưỡng tin cậy

# --- 2. CẤU HÌNH KÍCH THƯỚC CHUẨN ---
# Kích thước ảnh sau khi duỗi thẳng (giữ tỉ lệ tờ giấy)
STD_WIDTH = 800
STD_HEIGHT = 1100 # Giả định tỉ lệ ~A4

# --- 3. CẤU HÌNH CÁC VÙNG CẮT CHUẨN (ROI) ---
# Tọa độ (x, y, w, h) trên ảnh đã được duỗi thẳng (kích thước 800x1100)
# Bạn sẽ cần tinh chỉnh các con số này 1 LẦN DUY NHẤT
# bằng cách chạy code, xem ảnh "Anh da duoi thang" và ước lượng.
# (x, y, width, height)
ROI_STUDENT_ID = (445, 20, 80, 130) # (x, y, width, height) - VÍ DỤ
ROI_EXAM_CODE = (530, 15, 50, 130) # (x, y, width, height) - VÍ DỤ
ROI_ANSWERS_BLOCK1 = (100, 200, 100, 340)   # (x, y, w, h) - VÍ DỤ cho Khối 1 (Câu 1-25)
ROI_ANSWERS_BLOCK2 = (350, 200, 100, 340)  # (x, y, w, h) - VÍ DỤ cho Khối 2 (Câu 26-50)

# --- 4. CẤU HÌNH LƯỚI (GRID) ---
# Cấu hình này giúp diễn giải tọa độ của ô được tô thành số/chữ
GRID_CONFIG_ID_CODE = {
    'rows': 10,  # 10 hàng số từ 0 đến 9
    'cols_id': 6,    # SBD có 6 số
    'cols_code': 3 # Mã đề có 3 số
}
GRID_CONFIG_ANSWERS = {
    'rows': 25,  # 25 câu trong MỘT khối
    'cols': 4,   # 4 cột (A, B, C, D)
    'labels': ['A', 'B', 'C', 'D']
}

# --- 5. ĐÁP ÁN ĐÚNG ---
ANSWER_KEY = {
    "129": {
        1: 'A', 2: 'B', 3: 'C', 4: 'D', 5: 'A', 6: 'B', 7: 'C', 8: 'D', 9: 'A', 10: 'B',
        11: 'A', 12: 'B', 13: 'C', 14: 'D', 15: 'A', 16: 'B', 17: 'C', 18: 'D', 19: 'A', 20: 'B',
        21: 'A', 22: 'B', 23: 'C', 24: 'D', 25: 'A', 26: 'A', 27: 'B', 28: 'C', 29: 'D', 30: 'A',
        31: 'B', 32: 'C', 33: 'D', 34: 'A', 35: 'B', 36: 'C', 37: 'D', 38: 'A', 39: 'B', 40: 'C',
        41: 'D', 42: 'A', 43: 'B', 44: 'C', 45: 'D', 46: 'A', 47: 'B', 48: 'C', 49: 'D', 50: 'A'
    },
    # Thêm các mã đề khác nếu cần
}
TOTAL_QUESTIONS = 50
# -------------------------------------------------------------------
# KẾT THÚC CẤU HÌNH
# -------------------------------------------------------------------

def find_and_warp_sheet(image, std_width, std_height):
    """
    Tìm 4 dấu mốc vuông và duỗi thẳng ảnh về kích thước chuẩn.
    """
    print("Đang tìm 4 dấu mốc để duỗi thẳng ảnh...")
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    
    # Ngưỡng nhị phân hóa ảnh để làm nổi bật 4 dấu mốc đen
    _, thresh = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    
    # Tìm tất cả các đường viền bên ngoài
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        print("Lỗi tiền xử lý: Không tìm thấy contour nào.")
        return None

    # Lọc các contour để tìm 4 dấu mốc
    # Giả định các dấu mốc là các contour vuông, có diện tích hợp lý
    min_area = (image.shape[0] * image.shape[1]) / 6000 # Điều chỉnh nếu cần
    max_area = (image.shape[0] * image.shape[1]) / 200 # Điều chỉnh nếu cần
    marker_centers = []
    
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if min_area < area < max_area:
            peri = cv2.arcLength(cnt, True)
            approx = cv2.approxPolyDP(cnt, 0.04 * peri, True)
            # Dấu mốc là hình vuông (4 cạnh)
            if len(approx) == 4:
                M = cv2.moments(cnt)
                if M["m00"] != 0:
                    cx = int(M["m10"] / M["m00"])
                    cy = int(M["m01"] / M["m00"])
                    marker_centers.append((cx, cy))

    if len(marker_centers) < 4:
        print(f"Lỗi tiền xử lý: Chỉ tìm thấy {len(marker_centers)} dấu mốc, cần 4.")
        print("Hãy thử điều chỉnh min_area/max_area hoặc kiểm tra ảnh đầu vào.")
        return None
    
    print(f"Đã tìm thấy {len(marker_centers)} dấu mốc. Đang chọn 4 góc...")
    
    # Sắp xếp 4 tâm này để tìm 4 góc
    centers = np.array(marker_centers, dtype=np.float32)
    
    # Top-Left (có tổng x+y nhỏ nhất)
    sum_xy = centers.sum(axis=1)
    tl = centers[np.argmin(sum_xy)]
    
    # Bottom-Right (có tổng x+y lớn nhất)
    br = centers[np.argmax(sum_xy)]
    
    # Top-Right (có hiệu x-y lớn nhất)
    diff_xy = np.diff(centers, axis=1) # (x-y)
    tr = centers[np.argmax(diff_xy)]
    
    # Bottom-Left (có hiệu x-y nhỏ nhất)
    bl = centers[np.argmin(diff_xy)]
    
    src_points = np.array([tl, tr, br, bl], dtype=np.float32)
    
    dst_points = np.array([
        [0, 0],
        [std_width, 0],
        [std_width, std_height],
        [0, std_height]
    ], dtype=np.float32)
    
    # Tạo ma trận biến đổi và duỗi ảnh
    M = cv2.getPerspectiveTransform(src_points, dst_points)
    warped_image = cv2.warpPerspective(image, M, (std_width, std_height))
    
    print("Đã duỗi thẳng ảnh thành công.")
    return warped_image

# --- Các hàm trợ giúp (giữ nguyên từ code trước) ---

def resize_for_display(image, max_height=800):
    """Thay đổi kích thước ảnh để hiển thị vừa vặn."""
    try:
        h, w = image.shape[:2]
        if h > max_height:
            scale = max_height / h
            return cv2.resize(image, (int(w * scale), max_height))
        return image
    except Exception:
        return np.zeros((max_height, max_height, 3), dtype=np.uint8)


def draw_detections(image, detections, class_names):
    """Vẽ các bounding box lên ảnh."""
    output_image = image.copy()
    for box in detections.boxes:
        x1, y1, x2, y2 = [int(i) for i in box.xyxy[0]]
        conf = box.conf[0]
        cls_id = int(box.cls[0])
        label = f"{class_names[cls_id]}: {conf:.2f}"
        
        cv2.rectangle(output_image, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(output_image, label, (x1, y1 - 10), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    return output_image

def interpret_id_code_grid(detections, grid_config, num_cols):
    """Diễn giải các ô được tô thành chuỗi Mã SV hoặc Mã Đề."""
    H, W = detections.orig_shape[:2]
    num_rows = grid_config['rows']
    
    col_width = W / num_cols
    row_height = H / num_rows
    
    col_selections = {}

    for box in detections.boxes:
        center_x = (box.xyxy[0][0] + box.xyxy[0][2]) / 2
        center_y = (box.xyxy[0][1] + box.xyxy[0][3]) / 2
        
        col_index = int(center_x / col_width)
        row_index = int(center_y / row_height) # Đây chính là con số (0-9)
        
        if col_index not in col_selections:
            col_selections[col_index] = []
        col_selections[col_index].append(row_index)

    result_str = ""
    for i in range(num_cols):
        if i not in col_selections:
            result_str += "X" # X = Bỏ trống
        elif len(col_selections[i]) > 1:
            result_str += "E" # E = Lỗi (tô 2 số)
        else:
            result_str += str(col_selections[i][0])
            
    return result_str

def interpret_answer_grid(detections, grid_config):
    """Diễn giải các ô được tô thành từ điển đáp án {câu: đáp án}."""
    H, W = detections.orig_shape[:2]
    num_rows = grid_config['rows']
    num_cols = grid_config['cols']
    labels = grid_config['labels']
    
    col_width = W / num_cols
    row_height = H / num_rows
    
    row_selections = {}

    for box in detections.boxes:
        center_x = (box.xyxy[0][0] + box.xyxy[0][2]) / 2
        center_y = (box.xyxy[0][1] + box.xyxy[0][3]) / 2
        
        col_index = int(center_x / col_width)
        row_index = int(center_y / row_height) # Đây là số thứ tự câu (từ 0)
        
        if row_index not in row_selections:
            row_selections[row_index] = []
        row_selections[row_index].append(col_index)

    student_answers = {}
    for i in range(num_rows):
        question_num = i + 1 # Câu hỏi bắt đầu từ 1
        
        if i not in row_selections:
            student_answers[question_num] = "Bỏ trống"
        elif len(row_selections[i]) > 1:
            student_answers[question_num] = "Phạm quy" # Tô nhiều đáp án
        else:
            col_idx = row_selections[i][0]
            if 0 <= col_idx < len(labels):
                student_answers[question_num] = labels[col_idx]
            else:
                student_answers[question_num] = "Lỗi" # Lỗi logic
                
    return student_answers

def grade_answers(student_answers, correct_key):
    """Chấm điểm bài làm."""
    if not correct_key:
        return 0, 0
        
    score = 0
    correct_count = 0
    points_per_q = 10.0 / TOTAL_QUESTIONS

    for q_num, student_ans in student_answers.items():
        if q_num in correct_key:
            correct_ans = correct_key[q_num]
            if student_ans == correct_ans:
                correct_count += 1
                
    score = correct_count * points_per_q
    return score, correct_count

def draw_answer_feedback(image, student_answers, correct_key, grid_config):
    """Vẽ phản hồi Đúng/Sai lên ảnh đáp án."""
    output_image = image.copy()
    H, W = output_image.shape[:2]
    num_rows = grid_config['rows']
    num_cols = grid_config['cols']
    labels = grid_config['labels']
    
    col_width = W / num_cols
    row_height = H / num_rows

    for q_num, student_ans in student_answers.items():
        if q_num not in correct_key:
            continue
            
        correct_ans = correct_key[q_num]
        row_index = q_num - 1
        
        y = int((row_index + 0.5) * row_height)
        
        if student_ans == correct_ans:
            if student_ans in labels:
                col_index = labels.index(student_ans)
                x = int((col_index + 0.5) * col_width)
                cv2.circle(output_image, (x, y), int(row_height*0.4), (0, 255, 0), 2) # Xanh lá
        
        elif student_ans != "Bỏ trống" and student_ans != "Phạm quy":
            if student_ans in labels:
                col_index_student = labels.index(student_ans)
                x_student = int((col_index_student + 0.5) * col_width)
                cv2.line(output_image, (x_student-5, y-5), (x_student+5, y+5), (0, 0, 255), 2)
                cv2.line(output_image, (x_student-5, y+5), (x_student+5, y-5), (0, 0, 255), 2)
            
            if correct_ans in labels:
                col_index_correct = labels.index(correct_ans)
                x_correct = int((col_index_correct + 0.5) * col_width)
                cv2.circle(output_image, (x_correct, y), int(row_height*0.4), (0, 255, 0), 2) # Xanh lá
                
    return output_image

# --- HÀM MAIN (ĐÃ CẬP NHẬT) ---

def main():
    # 1. Tải Model
    try:
        model = YOLO(MODEL_PATH)
        class_names = model.names
        print(f"Tải model thành công. Các lớp: {class_names}")
    except Exception as e:
        print(f"Lỗi tải model '{MODEL_PATH}'. Lỗi: {e}")
        return

    # 2. Tải ảnh gốc
    if not os.path.exists(INPUT_IMAGE_PATH):
        print(f"Lỗi: Không tìm thấy ảnh '{INPUT_IMAGE_PATH}'")
        return
    
    original_image = cv2.imread(INPUT_IMAGE_PATH)
    if original_image is None:
        print("Lỗi: Không thể đọc ảnh.")
        return

    # 3. TIỀN XỬ LÝ: Tìm và duỗi thẳng ảnh
    warped_image = find_and_warp_sheet(original_image, STD_WIDTH, STD_HEIGHT)
    
    if warped_image is None:
        print("Không thể xử lý ảnh. Đang thoát...")
        return
    
    # Khởi tạo các biến kết quả
    student_id_str = "Chưa có"
    exam_code_str = "Chưa có"
    student_answers_final = {}
    score = 0
    
    img_id_result = None
    img_code_result = None
    img_answers_results_list = []

    # 4. Xử lý Mã Sinh Viên (trên ảnh đã duỗi thẳng)
    if ROI_STUDENT_ID:
        try:
            x, y, w, h = ROI_STUDENT_ID
            img_id = warped_image[y:y+h, x:x+w]
            detections_id = model.predict(img_id, conf=CONFIDENCE_THRESHOLD, verbose=False)[0]
            student_id_str = interpret_id_code_grid(detections_id, GRID_CONFIG_ID_CODE, GRID_CONFIG_ID_CODE['cols_id'])
            img_id_result = draw_detections(img_id, detections_id, class_names)
            print(f"Xử lý SBD thành công: {student_id_str}")
        except Exception as e:
            print(f"Lỗi khi xử lý SBD: {e}")
            img_id_result = np.zeros((100, 100, 3), dtype=np.uint8)

    # 5. Xử lý Mã Đề (trên ảnh đã duỗi thẳng)
    if ROI_EXAM_CODE:
        try:
            x, y, w, h = ROI_EXAM_CODE
            img_code = warped_image[y:y+h, x:x+w]
            detections_code = model.predict(img_code, conf=CONFIDENCE_THRESHOLD, verbose=False)[0]
            exam_code_str = interpret_id_code_grid(detections_code, GRID_CONFIG_ID_CODE, GRID_CONFIG_ID_CODE['cols_code'])
            img_code_result = draw_detections(img_code, detections_code, class_names)
            print(f"Xử lý Mã Đề thành công: {exam_code_str}")
        except Exception as e:
            print(f"Lỗi khi xử lý Mã Đề: {e}")
            img_code_result = np.zeros((100, 100, 3), dtype=np.uint8)

    # 6. Cắt, Xử lý và Chấm Đáp Án (trên ảnh đã duỗi thẳng)
    answer_rois = [ROI_ANSWERS_BLOCK1, ROI_ANSWERS_BLOCK2]
    question_offset = 0
    correct_key = ANSWER_KEY.get(exam_code_str, None)
    
    if correct_key:
        print(f"Đang chấm bài cho mã đề: {exam_code_str}")
    else:
        print(f"Lỗi: Không tìm thấy đáp án cho mã đề '{exam_code_str}'")

    for i, roi_config in enumerate(answer_rois):
        if roi_config is None:
            question_offset += GRID_CONFIG_ANSWERS['rows']
            continue
        
        print(f"\n--- Đang xử lý Khối Đáp Án {i+1} ---")
        try:
            x, y, w, h = roi_config
            img_answers = warped_image[y:y+h, x:x+w]
            detections_answers = model.predict(img_answers, conf=CONFIDENCE_THRESHOLD, verbose=False)[0]
            answers_block = interpret_answer_grid(detections_answers, GRID_CONFIG_ANSWERS)
            
            for q_num, ans in answers_block.items():
                final_q_num = q_num + question_offset
                student_answers_final[final_q_num] = ans
            
            print(f"Đã diễn giải xong từ câu {question_offset + 1} đến {question_offset + GRID_CONFIG_ANSWERS['rows']}")

            if correct_key:
                block_correct_key = {}
                for q_num in range(1, GRID_CONFIG_ANSWERS['rows'] + 1):
                    final_q_num = q_num + question_offset
                    if final_q_num in correct_key:
                        block_correct_key[q_num] = correct_key[final_q_num]
                
                img_result = draw_answer_feedback(img_answers, answers_block, block_correct_key, GRID_CONFIG_ANSWERS)
                img_answers_results_list.append(img_result)
            else:
                img_result = draw_detections(img_answers, detections_answers, class_names)
                img_answers_results_list.append(img_result)
            
            question_offset += GRID_CONFIG_ANSWERS['rows']

        except Exception as e:
            print(f"Lỗi khi xử lý Khối Đáp Án {i+1}: {e}")
            img_answers_results_list.append(np.zeros((100, 100, 3), dtype=np.uint8))

    if correct_key:
        score, correct_count = grade_answers(student_answers_final, correct_key)
        print(f"\n--- TỔNG KẾT CHẤM ĐIỂM ---")
        print(f"Số câu đúng: {correct_count}/{TOTAL_QUESTIONS}")
        print(f"Điểm số: {score:.2f} / 10.0")
    else:
        score = 0.0

    # 7. In kết quả tổng thể lên ảnh gốc
    # Chúng ta sẽ in lên góc trên bên trái của ảnh GỐC
    img_main_result = original_image.copy()
    text_score = f"Diem: {score:.2f}"
    text_id = f"Ma SV: {student_id_str}"
    text_code = f"Ma De: {exam_code_str}"
    
    # Vẽ một hình chữ nhật đen làm nền để chữ nổi bật
    cv2.rectangle(img_main_result, (10, 10), (450, 160), (0, 0, 0), -1) 
    cv2.putText(img_main_result, text_score, (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 3)
    cv2.putText(img_main_result, text_id, (20, 100), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 3)
    cv2.putText(img_main_result, text_code, (20, 150), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 3)

    # 8. Hiển thị tất cả các cửa sổ
    cv2.imshow("1. Ket Qua Tong The (Tren Anh Goc)", resize_for_display(img_main_result, 800))
    cv2.imshow("2. Anh da duoi thang", resize_for_display(warped_image, 800))
    
    if img_id_result is not None:
        cv2.imshow("3. Xu Ly - Ma Sinh Vien", resize_for_display(img_id_result, 400))
    if img_code_result is not None:
        cv2.imshow("4. Xu Ly - Ma De", resize_for_display(img_code_result, 400))
    
    for i, img_result in enumerate(img_answers_results_list):
        cv2.imshow(f"5.{i+1} Xu Ly - Dap An Khoi {i+1}", resize_for_display(img_result, 800))

    print("\n--- HOAN THANH ---")
    print("Nhấn phím bất kỳ trên cửa sổ ảnh để thoát...")
    cv2.waitKey(0)
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()

Tải model thành công. Các lớp: {0: 'selected'}
Đang tìm 4 dấu mốc để duỗi thẳng ảnh...
Đã tìm thấy 7 dấu mốc. Đang chọn 4 góc...
Đã duỗi thẳng ảnh thành công.
Xử lý SBD thành công: XXXXXX
Xử lý Mã Đề thành công: XXX
Lỗi: Không tìm thấy đáp án cho mã đề 'XXX'

--- Đang xử lý Khối Đáp Án 1 ---
Đã diễn giải xong từ câu 1 đến 25

--- Đang xử lý Khối Đáp Án 2 ---
Đã diễn giải xong từ câu 26 đến 50

--- HOAN THANH ---
Nhấn phím bất kỳ trên cửa sổ ảnh để thoát...


test 2

In [ ]:
import cv2
import os
import numpy as np
from ultralytics import YOLO

# --- CẤU HÌNH ---
MODEL_PATH = 'best.pt'
INPUT_IMAGE_PATH = r"D:\data\phieu_50.jpg"
CONFIDENCE_THRESHOLD = 0.4
STD_WIDTH, STD_HEIGHT = 800, 1100

# Định nghĩa ROI dựa trên ảnh 800x1100
ROI_STUDENT_ID = (445, 20, 80, 130)  # SBD
ROI_EXAM_CODE = (530, 15, 50, 130)   # Mã đề
ROI_ANSWERS_BLOCK1 = (100, 200, 100, 340)  # Câu 1-25
ROI_ANSWERS_BLOCK2 = (350, 200, 100, 340)  # Câu 26-50

GRID_ANSWERS = {'rows': 25, 'cols': 4, 'labels': ['A', 'B', 'C', 'D']}
GRID_CODE = {'rows': 10, 'cols': 3} # 10 hàng (0-9), 3 cột cho Mã đề

ANSWER_KEY = {
    "129": {i: 'A' for i in range(1, 51)}, # Ví dụ bộ đáp án mã 129
    "001": {i: 'B' for i in range(1, 51)}  # Ví dụ bộ đáp án mã 001
}

# --- HÀM TRỢ GIÚP ÁNH XẠ ---

def get_centroid_mapping(roi_rect, row_idx, col_idx, grid_cfg):
    """Tính toán tọa độ tâm tuyệt đối trên ảnh warped từ chỉ số hàng/cột."""
    rx, ry, rw, rh = roi_rect
    cell_w = rw / grid_cfg['cols']
    cell_h = rh / grid_cfg['rows']
    
    # +0.5 để lấy điểm chính giữa ô
    cx = int(rx + (col_idx + 0.5) * cell_w)
    cy = int(ry + (row_idx + 0.5) * cell_h)
    return cx, cy

def find_and_warp(image):
    """Tìm 4 mốc và duỗi ảnh (giữ nguyên logic tìm contour của bạn)."""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(cv2.GaussianBlur(gray, (5, 5), 0), 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    cnts, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    markers = []
    for c in cnts:
        if (image.shape[0]*image.shape[1])/6000 < cv2.contourArea(c):
            approx = cv2.approxPolyDP(c, 0.04 * cv2.arcLength(c, True), True)
            if len(approx) == 4:
                M = cv2.moments(c)
                if M["m00"] != 0:
                    markers.append((int(M["m10"]/M["m00"]), int(M["m01"]/M["m00"])))
    
    if len(markers) < 4: return None
    centers = np.array(markers, dtype="float32")
    s = centers.sum(axis=1)
    diff = np.diff(centers, axis=1)
    rect = np.array([centers[np.argmin(s)], centers[np.argmin(diff)], 
                     centers[np.argmax(s)], centers[np.argmax(diff)]], dtype="float32")
    
    dst = np.array([[0,0], [STD_WIDTH,0], [STD_WIDTH,STD_HEIGHT], [0,STD_HEIGHT]], dtype="float32")
    return cv2.warpPerspective(image, cv2.getPerspectiveTransform(rect, dst), (STD_WIDTH, STD_HEIGHT))

# --- CHƯƠNG TRÌNH CHÍNH ---

def main():
    model = YOLO(MODEL_PATH)
    original = cv2.imread(INPUT_IMAGE_PATH)
    warped = find_and_warp(original)
    if warped is None: return

    canvas = warped.copy() # Ảnh dùng để vẽ đối chiếu
    
    # 1. NHẬN DIỆN MÃ ĐỀ (Cực kỳ quan trọng để lấy Key)
    x, y, w, h = ROI_EXAM_CODE
    roi_code = warped[y:y+h, x:x+w]
    res_code = model.predict(roi_code, conf=CONFIDENCE_THRESHOLD, verbose=False)[0]
    
    code_digits = {}
    for box in res_code.boxes:
        cx, cy = (box.xyxy[0][0] + box.xyxy[0][2])/2, (box.xyxy[0][1] + box.xyxy[0][3])/2
        col = int(cx / (w / GRID_CODE['cols']))
        row = int(cy / (h / GRID_CODE['rows']))
        code_digits[col] = str(row)
    
    exam_code = "".join([code_digits.get(i, "X") for i in range(GRID_CODE['cols'])])
    print(f"Mã đề nhận diện được: {exam_code}")

    # 2. LẤY ĐÁP ÁN CHUẨN
    correct_key = ANSWER_KEY.get(exam_code)
    if not correct_key:
        print(f"CẢNH BÁO: Không có đáp án cho mã đề {exam_code}. Dừng chấm điểm.")
        return

    # 3. NHẬN DIỆN VÀ CHẤM ĐÁP ÁN
    ans_blocks = [(ROI_ANSWERS_BLOCK1, 0), (ROI_ANSWERS_BLOCK2, 25)]
    student_results = {}

    for roi_rect, offset in ans_blocks:
        rx, ry, rw, rh = roi_rect
        roi_img = warped[ry:ry+rh, rx:rx+rw]
        res_ans = model.predict(roi_img, conf=CONFIDENCE_THRESHOLD, verbose=False)[0]
        
        for box in res_ans.boxes:
            bcx, bcy = (box.xyxy[0][0] + box.xyxy[0][2])/2, (box.xyxy[0][1] + box.xyxy[0][3])/2
            col_idx = int(bcx / (rw / 4))
            row_idx = int(bcy / (rh / 25))
            
            q_num = row_idx + 1 + offset
            ans_label = GRID_ANSWERS['labels'][col_idx]
            student_results[q_num] = ans_label

            # ÁNH XẠ VÀ VẼ ĐỐI CHIẾU
            # Vẽ vòng tròn xanh cho lựa chọn của sinh viên
            cx, cy = get_centroid_mapping(roi_rect, row_idx, col_idx, GRID_ANSWERS)
            
            if q_num in correct_key:
                if ans_label == correct_key[q_num]:
                    cv2.circle(canvas, (cx, cy), 12, (0, 255, 0), 2) # Đúng -> Xanh lá
                else:
                    cv2.drawMarker(canvas, (cx, cy), (0, 0, 255), cv2.MARKER_TILTED_CROSS, 15, 2) # Sai -> X đỏ
                    # Vẽ đáp án đúng bằng vòng tròn vàng
                    corr_col = GRID_ANSWERS['labels'].index(correct_key[q_num])
                    ccx, ccy = get_centroid_mapping(roi_rect, row_idx, corr_col, GRID_ANSWERS)
                    cv2.circle(canvas, (ccx, ccy), 12, (0, 255, 255), 2)

    # 4. TỔNG KẾT
    correct_count = sum(1 for q, a in student_results.items() if correct_key.get(q) == a)
    score = (correct_count / 50) * 10

    # Vẽ Header thông tin lên ảnh kết quả
    cv2.rectangle(canvas, (0, 0), (STD_WIDTH, 140), (230, 230, 230), -1)
    cv2.putText(canvas, f"MA DE: {exam_code}", (30, 50), 0, 1, (0,0,0), 2)
    cv2.putText(canvas, f"DIEM: {score:.2f}", (30, 110), 0, 1.5, (255, 0, 0), 3)
    cv2.putText(canvas, f"DUNG: {correct_count}/50", (450, 110), 0, 1, (0, 150, 0), 2)

    # Hiển thị ảnh đối chiếu cuối cùng
    cv2.imshow("DOI CHIEU KET QUA", canvas)
    cv2.imwrite("ket_qua_doi_chieu.jpg", canvas)
    cv2.waitKey(0)

if __name__ == "__main__":
    main()